## Historia de Usuario

**Revendedores y dropshippers:** evalúan rápido cientos de productos de la competencia para elegir qué vender.

**Analistas de mercado:** estiman la demanda de productos cuando Amazon no muestra el dato de compras. Recuerda que 1.124 productos de tu dataset no lo tenían.

| Dato | Qué es | ¿Lo usa el modelo para predecir? |
|---|---|---|
| ASIN | Identificador ("cuál es") | No: no contiene información de demanda |
| Reseñas | Característica ("cómo es") | Sí: variable principal |
| Precio de lista | Característica ("cómo es") | Sí: segunda variable |


| Uso | Para qué le sirve al revendedor |
|---|---|
| **Identificar** | Saber qué producto evaluó y guardar el historial ("¿qué resultado tuvo este audífono la semana pasada?") |
| **Autocompletar** | Ingresa solo el ASIN y la app trae sola las reseñas y el precio, sin tipear ni equivocarse de precio |
| **Evaluar en lote** | Pega una lista de 200 ASIN y recibe una tabla ordenada por probabilidad de alta demanda |

| | Versión 1: manual | Versión 2: automática |
|---|---|---|
| **Input** | ASIN + reseñas + precio de lista | Solo el ASIN (o una lista de ASIN) |
| **Cómo obtiene los datos** | El usuario los copia desde Amazon | La app los consulta por API |
| **Complejidad** | Baja: se puede hacer con Streamlit en 1 día | Alta: requiere acceso a la API oficial de Amazon o a un proveedor de datos pagado (por ejemplo, Keepa) |
| **Riesgo** | Errores de tipeo; ingresar el precio rebajado en vez del de lista | Costo de la API; extraer datos de Amazon sin permiso infringe sus términos de uso |

| Caso | Por qué la app sirve |
|---|---|
| **Productos sin el dato de compras** | En tu dataset, 1.124 productos (12,7%) no lo mostraban. Ahí la app es la única estimación disponible |
| **Evaluación masiva** | Revisar 200 páginas a mano toma horas; la app ordena la lista en segundos |
| **Datos exportados** | Los revendedores trabajan con planillas de herramientas como Keepa, que traen reseñas y precio pero no siempre el dato de compras |

| Evitar | Usar |
|---|---|
| "Pronostica si el producto tendrá buena demanda" | "Estima si el producto tiene hoy un perfil de alta demanda" |
| "Predice cuánto venderá" | "Clasifica si vende 400 o más unidades al mes" |

Especificación del prototipo (versión 1)

| Elemento | Definición |
|---|---|
| **Usuario** | Revendedor en Amazon que evalúa si conviene vender un producto |
| **Inputs** | ASIN (identificación), número de reseñas, precio de lista en US$ |
| **Validaciones** | Reseñas entre 5 y 94.000; precio entre US$7 y US$1.700; solo electrónica. Fuera de rango: aviso "resultado poco confiable" |
| **Modelo** | KNN (k=201) con reseñas y precio, entrenado con los 7.289 productos |
| **Output 1** | Probabilidad de alta demanda (%) |
| **Output 2** | Semáforo: 🟢 ≥ 60% alta · 🟡 40–60% zona gris · 🔴 < 40% baja |
| **Output 3** | Explicación frente al perfil típico (más de 1.500 reseñas y precio bajo US$110) |
| **Advertencias fijas** | Acierta 3 de cada 4 productos · solo electrónica en Amazon EE.UU. · no sirve para productos sin reseñas · identifica, no recomienda |
| **Herramienta sugerida** | Streamlit (Python): un solo archivo, corre en el navegador |

# Paso 11 -- Guardar el modelo para la app

In [ ]:

import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. Los mismos 7.289 productos del modelo (1 fila por producto, con reseñas, precio y compras)
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
datos = pd.read_csv(BASE / "data" / "procesado" / "amazon_electronics_cleaned_with_asin.csv", parse_dates=["data_collected_at"])
productos = datos.sort_values("data_collected_at")
productos = productos[productos["asin"].isna() | ~productos.duplicated("asin", keep="last")]
modelo = productos.dropna(subset=["total_reviews", "original_price", "product_rating", "purchased_last_month"])

# 2. Solo reseñas y precio: el rating no aporta (AUC 0,826 sin él vs 0,828 con él) y así la app pide un dato menos
X_app = pd.DataFrame({"log_reseñas": np.log10(modelo["total_reviews"]), "log_precio": np.log10(modelo["original_price"])})
y_app = (modelo["purchased_last_month"] >= 400).astype(int)

# 3. Entrenar con TODOS los productos (la evaluación ya se hizo) y guardar en un archivo
modelo_app = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=201)).fit(X_app, y_app)
(BASE / "modelos").mkdir(exist_ok=True)
joblib.dump(modelo_app, BASE / "modelos" / "modelo_demanda.joblib")
print(f"Modelo guardado en modelos/modelo_demanda.joblib | entrenado con {len(X_app):,} productos")

Modelo guardado en modelos/modelo_demanda.joblib | entrenado con 7,289 productos


Qué hace, en simple:

Entrena con todos los productos (100%). El 80/20 servía para medir el modelo, y eso ya está hecho. Para usarlo en la práctica conviene que aprenda de todos los datos.
Guarda el modelo en un archivo (.joblib, de unos 370 KB), como quien guarda una planilla. La app abre ese archivo y no tiene que volver a entrenar cada vez.

| Caso | ASIN | Reseñas | Precio de lista | Debe mostrar |
|---|---|---:|---:|---|
| 1 | B0DSCFVFZ5 | 5000 | 30 | 🟢 Alta demanda · 81% |
| 2 | (vacío) | 1500 | 80 | 🟡 Zona gris · 53% |
| 3 | abc | 300 | 200 | Aviso de ASIN + 🔴 Baja demanda · 32% |
| 4 | (vacío) | 2 | 5000 | Aviso "fuera de rango" + 🔴 Baja demanda · 4% |

In [2]:
# Paso 12 -- Análisis de sensibilidad: ¿cambia la conclusión si los productos sin dato de compras se cuentan como baja demanda?
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
datos = pd.read_csv(BASE / "data" / "procesado" / "amazon_electronics_cleaned_with_asin.csv", parse_dates=["data_collected_at"])
productos = datos.sort_values("data_collected_at")
productos = productos[productos["asin"].isna() | ~productos.duplicated("asin", keep="last")]   # 1 fila por producto
con_x = productos.dropna(subset=["total_reviews", "original_price", "product_rating"])           # tienen las 3 variables X

escenarios = {   # nombre: productos usados
    "Base: solo productos con dato de compras": con_x[con_x["purchased_last_month"].notna()],
    "Sensibilidad: sin dato de compras = baja demanda": con_x}

SEMILLA = 42
filas = {}
for nombre, d in escenarios.items():
    X = pd.DataFrame({"log_reseñas": np.log10(d["total_reviews"]), "log_precio": np.log10(d["original_price"]),
                      "rating": d["product_rating"]})
    y = (d["purchased_last_month"].fillna(0) >= 400).astype(int)          # NaN -> 0 -> baja demanda
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEMILLA)
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=201))
    prob = lambda cols: knn.fit(X_train[cols], y_train).predict_proba(X_test[cols])[:, 1]
    auc = lambda cols: roc_auc_score(y_test, prob(cols))
    p3, pr = prob(list(X.columns)), prob(["log_reseñas"])
    auc_3, auc_r = roc_auc_score(y_test, p3), roc_auc_score(y_test, pr)
    rng = np.random.default_rng(SEMILLA)                                   # misma prueba bootstrap del Paso 10
    dif = [roc_auc_score(y_test.iloc[i], p3[i]) - roc_auc_score(y_test.iloc[i], pr[i])
           for i in (rng.integers(0, len(y_test), len(y_test)) for _ in range(2000))]
    filas[nombre] = {"Productos": len(d), "% del total": len(d) / len(productos) * 100, "% alta demanda": y.mean() * 100,
                     "AUC solo reseñas": auc_r, "AUC reseñas + precio + rating": auc_3, "Mejora": auc_3 - auc_r,
                     "IC 95% inferior": np.percentile(dif, 2.5), "valor p": np.mean(np.array(dif) <= 0),
                     "AUC sin precio": auc(["log_reseñas", "rating"]), "AUC sin rating": auc(["log_reseñas", "log_precio"])}
display(pd.DataFrame(filas).T.round(3))

,Productos,% del total,% alta demanda,AUC solo reseñas,AUC reseñas + precio + rating,Mejora,IC 95% inferior,valor p,AUC sin precio,AUC sin rating
Base: solo productos con dato de compras,7289.0,82.595,53.299,0.799,0.833,0.034,0.020,0.0,0.799,0.831
Sensibilidad: sin dato de compras = baja demanda,8333.0,94.425,46.622,0.772,0.799,0.027,0.015,0.0,0.783,0.788
